## Outline analysis within the square-root velocity framework 

This notebook shows the steps to perform an outline analysis with a set of correspondent coordinates in the square-root velocity framework. 
### Imports

In [ ]:
import pandas as pd
import numpy as np
import os
import tqdm
from copy import deepcopy
from SqST_Functions import align, formatData, geodDistance

### Data Preparation

Set the paths needed to the expected input files:
1. The index file is expected to be a data frame with the columns "Index", "Filename" and "Species". Further columns with further categories are possible.
2. Outlines provides the CSV with the outlines coordinates for each specimen listed in the index file. Expected columns are "Filename", "x" and "y".
3. The out_path determines the saving location for all generated output files

In [ ]:
outlines = pd.read_csv('PATH') # Insert path to the table containing outlines coordinates
index = pd.read_csv('PATH') # Insert path to the index file for the dataset
out_path = 'PATH' # Insert path to the designated output folder

Merge outline coordinates with the index file for analysis of groups

In [ ]:
outlines_table = outlines.merge(index, on='Filename', how='right')
outlines_table= outlines_table.drop(['Index'], axis=1)
all_groups = np.unique(list(outlines_table['Species']))

### Karcher Mean and tPCA

Calculation of the Karcher Mean for each group, here species, in the dataset and the complete sample. Additionally the coordinates for each group within the tangens principal component analysis are calculated.

In [ ]:
all_km_rows= []

for group in tqdm.tqdm(all_groups):
    # Filter dataset for group
    newdf = outlines_table[outlines_table['Species'].isin([group])]
    names = list(pd.unique(newdf['Filename']))
    beta = formatData(newdf,varName='Filename', xName='x', yName='y')
    # Compute Karcher mean / tPCA:
    algn,km,tpca,_,_ = align(beta,returnDF=1, pd_plots=False)
    
    # Save Karcher mean for each group
    newname = str(group)
    km.to_csv(out_path + 'KM_' +newname+ '.csv',index=False)
    
    # Collect all group means for a single DataFrame
    for xi, yi, in zip(km['x'], km['y']):
	    all_km_rows.append({
		    'Filename': newname,
		    'x': xi,
		    'y': yi
		})
	
    # Save tPCA coordinates for each group:
    tpca.index=names
    tpca.index.name='Filename'
    tpca.to_csv(out_path +'tPCA_' +newname+ '.csv')
    
# Create and save the DataFrame with all group means
colGroupMeans = pd.DataFrame(all_km_rows)
colGroupMeans.to_csv(out_path + 'YOUR_FILENAME.csv', index=False)

#### Karcher Mean and tPCA of Total Sample and the PC axes variances
Calculate the mean shape of the complete data set. Additionally the variances explained by the PC axes and the principal directions of these axes are extracted.

In [ ]:
files = list(pd.unique(outlines_table['Filename'])) 
beta = formatData(outlines_table,varName='Filename', xName='x', yName='y')

# Compute Karcher mean / tPCA incl variance per axis:
algnAll,kmAll,tpcaAll,var,var_df,pd_results = align(beta, returnDF=1, PC=10, pd_plots=True)

# Save Karcher mean:
kmAll.to_csv(out_path + 'YOUR_FILENAME.csv',index=False)

# Save tPCA 
tpcaAll.index=files
tpcaAll.index.name= 'Filename'
tpcaAll.to_csv(out_path + 'YOUR_FILENAME.csv')

# Save %variance for tPCA axes
var_df.to_csv(out_path + 'YOUR_FILENAME.csv', index=False)

The principal directions for the main PC axes have to be converted to a DataFrame before saving the CSV.

In [ ]:
pd_dict = {}
n_shapes_per_pc = 10

for i, shape_pair in enumerate(pd_results):
	pd_idx = (i // n_shapes_per_pc) + 1
	shape_idx = (i % n_shapes_per_pc) + 1
	x = shape_pair[0].squeeze()
	y = shape_pair[1].squeeze()
	pd_dict[f'PD_{pd_idx}_{shape_idx}_x'] = x
	pd_dict[f'PD_{pd_idx}_{shape_idx}_y'] = y
	
PDs = pd.DataFrame(pd_dict)
PDs.to_csv(out_path + 'YOUR_FILENAME.csv', index =False)

### Geodesic Distances based on Outlines

Calculate the geodesic distances between each specimen in the dataset.

In [ ]:
ot = deepcopy(outlines_table)
otnames = list(pd.unique(ot['Filename']))
n = len(otnames)
print(str(n)+' contours in dataset.')

distances = np.zeros((n,n))
errors = []

for i in tqdm.tqdm(range(0,n)):
    x1 = list(ot[ot['Filename']==otnames[i]]['x'])
    y1 = list(ot[ot['Filename']==otnames[i]]['y'])
    for j in range(i+1,n):
        x2 = list(ot[ot['Filename']==otnames[j]]['x'])
        y2 = list(ot[ot['Filename']==otnames[j]]['y'])
        try:
            d,_,_,_ = geodDistance(x1,y1,x2,y2,k=5)
        except:
            try:
                d,_,_,_ = geodDistance(x2,y2,x1,y1,k=5)
            except:
                print('Error for contours '+str(i)+' and '+str(j))
                errors.append([i,j])
                d = 100000
        distances[i,j] = d
        distances[j,i] = d

print('Computed distances between '+str(n)+' contours, with '+str(len(errors))+' errors.')

# Saving the distances in a pairwise table
geodesicDist = pd.DataFrame(distances,index=otnames)
geodesicDist.columns = otnames
geodesicDist.index.name='Filename'
geodesicDist.to_csv(out_path + 'YOUR_FILENAME.csv')

#### Compute geodesic distances between the mean shapes of the groups

In [ ]:
GM = deepcopy(colGroupMeans)
namesMeans = np.unique(list(GM['Filename']))
nMean = len(namesMeans)
print(str(nMean)+' contours in dataset.')

Mdistances = np.zeros((nMean,nMean))
errorsM = []

for i in tqdm.tqdm(range(0,nMean)):
    x1 = list(GM[GM['Filename']==namesMeans[i]]['x'])
    y1 = list(GM[GM['Filename']==namesMeans[i]]['y'])
    for j in range(i+1,nMean):
        x2 = list(GM[GM['Filename']==namesMeans[j]]['x'])
        y2 = list(GM[GM['Filename']==namesMeans[j]]['y'])
        try:
            d,_,_,_ = geodDistance(x1,y1,x2,y2,k=5)
        except:
            try:
                d,_,_,_ = geodDistance(x2,y2,x1,y1,k=5)
            except:
                print('Error for contours '+str(i)+' and '+str(j))
                errorsM.append([i,j])
                d = 100000
        Mdistances[i,j] = d
        Mdistances[j,i] = d

print('Computed distances between '+str(nMean)+' contours, with '+str(len(errorsM))+' errors.')

# Saving the distances between the means in a pairwise table
geodesicDistMeans = pd.DataFrame(Mdistances,index=namesMeans)
geodesicDistMeans.columns = namesMeans
geodesicDistMeans.index.name='Name'
geodesicDistMeans.to_csv(out_path + 'YOUR_FILENAME.csv')